# CityPulse AI — Model Training Notebook
**CET4973 — Spring 2026**  
**Team: Mohammed Imad, Ronny Yeap**

This notebook reproduces the full CityPulse AI pipeline:
1. Build the 2,000-record dataset (311-style features + scoring formula labels)
2. Train Logistic Regression baseline and Random Forest classifier
3. Generate every table and figure needed for the Experiments/Results/Discussion section

**Run all cells top to bottom. Every output is saved to `/content/` for download.**

## Cell 1 — Install & Import

In [ ]:
# All libraries are pre-installed in Colab — no pip needed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

import warnings
warnings.filterwarnings('ignore')

# Reproducibility — fixed seed used throughout
SEED = 42
np.random.seed(SEED)

print('Libraries loaded. NumPy:', np.__version__, '| scikit-learn imported OK')

## Cell 2 — Build the Dataset

We simulate 2,000 records that mirror what the real 311 + weather + facility pipeline produces.
Each borough gets realistic distributions based on:
- Kontokosta & Hong (2021): Bronx underreports → upweighted Complaints
- ACS 2022: borough-level elderly/disability estimates → Accessibility
- Complaint-type → Severity proxy (since Team 1 CV output not yet connected)

In [ ]:
N = 2000
rng = np.random.default_rng(SEED)

# --- Borough assignment (roughly proportional to 311 complaint volume) ---
boroughs = rng.choice(
    ['Brooklyn', 'Queens', 'Manhattan', 'Bronx', 'Staten Island'],
    size=N,
    p=[0.30, 0.25, 0.22, 0.16, 0.07]
)

# --- Complaint types and their Severity proxy ---
# Street Condition / Pothole → high severity; Street Light → low
complaint_types = rng.choice(
    ['Pothole', 'Street Condition', 'Sidewalk Condition', 'Street Light Condition'],
    size=N,
    p=[0.35, 0.30, 0.25, 0.10]
)
severity_map = {
    'Pothole': 4,
    'Street Condition': 3,
    'Sidewalk Condition': 2,
    'Street Light Condition': 1
}
# Add small noise (±0.5) to avoid perfectly deterministic severity
severity = np.array([severity_map[c] for c in complaint_types], dtype=float)
severity += rng.uniform(-0.5, 0.5, size=N)
severity = np.clip(severity, 1.0, 5.0).round(2)

# --- Weather multiplier (1.0 = dry, 1.5 = heavy rain / freezing) ---
# Winter months → higher risk; summer → lower
weather = rng.choice(
    [1.0, 1.1, 1.2, 1.3, 1.4, 1.5],
    size=N,
    p=[0.20, 0.20, 0.25, 0.15, 0.12, 0.08]
).astype(float)

# --- Impact: proximity to schools / hospitals (0–3) ---
# Manhattan and Brooklyn have denser facility coverage → slightly higher
impact_means = {
    'Brooklyn': 1.8, 'Queens': 1.5, 'Manhattan': 2.0,
    'Bronx': 1.6, 'Staten Island': 1.2
}
impact = np.array([
    np.clip(rng.normal(impact_means[b], 0.6), 0, 3)
    for b in boroughs
]).round(2)

# --- Complaints: 311 volume adjusted for underreporting (0–2) ---
# Bronx underreports → upweighted per Kontokosta & Hong (2021)
# Recency weighting applied inside the score (full/0.75/0.50)
complaint_means = {
    'Brooklyn': 1.2, 'Queens': 1.1, 'Manhattan': 1.4,
    'Bronx': 1.5,   # upweighted for underreporting
    'Staten Island': 0.8
}
complaints = np.array([
    np.clip(rng.normal(complaint_means[b], 0.4), 0, 2)
    for b in boroughs
]).round(2)

# --- Accessibility: vulnerable-user concentration (0–2, ACS 2022) ---
access_means = {
    'Brooklyn': 1.1, 'Queens': 1.0, 'Manhattan': 0.9,
    'Bronx': 1.3,    # highest elderly + disability rate
    'Staten Island': 1.0
}
accessibility = np.array([
    np.clip(rng.normal(access_means[b], 0.4), 0, 2)
    for b in boroughs
]).round(2)

# --- Priority Score formula ---
# Priority Score = (Severity × Weather) + Impact + Complaints + Accessibility
# Min = 1.0, Max = 14.5
priority_score = (severity * weather) + impact + complaints + accessibility
priority_score = priority_score.round(2)

# --- Tier labels (from proposal) ---
# Critical ≥ 11, High 7–10.99, Medium 4–6.99, Low < 4
def assign_tier(score):
    if score >= 11:  return 'Critical'
    elif score >= 7: return 'High'
    elif score >= 4: return 'Medium'
    else:            return 'Low'

tier = np.array([assign_tier(s) for s in priority_score])

# --- Assemble DataFrame ---
df = pd.DataFrame({
    'borough':        boroughs,
    'complaint_type': complaint_types,
    'severity':       severity,
    'weather':        weather,
    'impact':         impact,
    'complaints':     complaints,
    'accessibility':  accessibility,
    'priority_score': priority_score,
    'priority_tier':  tier
})

print('Dataset shape:', df.shape)
print('\nClass distribution:')
print(df['priority_tier'].value_counts())
print('\nSample records:')
df.head()

## Cell 3 — Train / Test Split & Scaling

In [ ]:
FEATURES = ['severity', 'weather', 'impact', 'complaints', 'accessibility']
LABEL    = 'priority_tier'

X = df[FEATURES].values
y = df[LABEL].values

# 80/20 stratified split — same proportions of all four tiers in both sets
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, np.arange(N),
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

# StandardScaler: subtract mean, divide by std — fit on train only
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)   # same scaler, no data leakage

print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print('\nTier counts in TEST set:')
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  {u}: {c}')

## Cell 4 — Train Logistic Regression (Baseline)

In [ ]:
# Logistic Regression — multinomial softmax, L2 regularization
# class_weight='balanced': penalizes minority class (Critical) more heavily
# solver='lbfgs': default, handles multinomial well on small datasets
# max_iter=1000: increased from default 100 to ensure convergence
lr = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    class_weight='balanced',
    max_iter=1000,
    random_state=SEED
)
lr.fit(X_train_sc, y_train)

y_pred_lr_train = lr.predict(X_train_sc)
y_pred_lr_test  = lr.predict(X_test_sc)

acc_lr_train = accuracy_score(y_train, y_pred_lr_train)
acc_lr_test  = accuracy_score(y_test,  y_pred_lr_test)
f1_lr_train  = f1_score(y_train, y_pred_lr_train, average='macro')
f1_lr_test   = f1_score(y_test,  y_pred_lr_test,  average='macro')

print('=== Logistic Regression ===')
print(f'Train Accuracy: {acc_lr_train:.4f} | Train Macro F1: {f1_lr_train:.4f}')
print(f'Test  Accuracy: {acc_lr_test:.4f}  | Test  Macro F1: {f1_lr_test:.4f}')
print()
print('Classification Report (Test Set):')
print(classification_report(y_test, y_pred_lr_test, digits=3))

## Cell 5 — Train Random Forest (Primary Classifier)

In [ ]:
# Random Forest — 100 trees, Gini impurity, balanced class weights
# n_estimators=100: stable performance without excessive compute on 2000 samples
# max_features='sqrt': each tree sees sqrt(5) ≈ 2 features per split (default)
# class_weight='balanced': same rationale as LR — prevent Critical being ignored
# random_state=42: fixed for reproducibility
rf = RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1   # use all available CPU cores
)
rf.fit(X_train, y_train)   # RF does not need scaled features (tree-based)

y_pred_rf_train = rf.predict(X_train)
y_pred_rf_test  = rf.predict(X_test)

acc_rf_train = accuracy_score(y_train, y_pred_rf_train)
acc_rf_test  = accuracy_score(y_test,  y_pred_rf_test)
f1_rf_train  = f1_score(y_train, y_pred_rf_train, average='macro')
f1_rf_test   = f1_score(y_test,  y_pred_rf_test,  average='macro')

print('=== Random Forest ===')
print(f'Train Accuracy: {acc_rf_train:.4f} | Train Macro F1: {f1_rf_train:.4f}')
print(f'Test  Accuracy: {acc_rf_test:.4f}  | Test  Macro F1: {f1_rf_test:.4f}')
print()
print('Classification Report (Test Set):')
print(classification_report(y_test, y_pred_rf_test, digits=3))

## Cell 6 — Figure 1: Confusion Matrix (Random Forest)

Save to `/content/fig1_confusion_matrix.png`

In [ ]:
CLASS_ORDER = ['Low', 'Medium', 'High', 'Critical']  # ascending priority

cm = confusion_matrix(y_test, y_pred_rf_test, labels=CLASS_ORDER)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_ORDER,
    yticklabels=CLASS_ORDER,
    linewidths=0.5, linecolor='gray',
    ax=ax
)
ax.set_xlabel('Predicted Label', fontsize=12, labelpad=8)
ax.set_ylabel('True Label', fontsize=12, labelpad=8)
ax.set_title('Figure 1: Confusion Matrix — Random Forest (Test Set, n=400)', fontsize=11, pad=10)
ax.tick_params(axis='both', labelsize=10)
plt.tight_layout()
plt.savefig('/content/fig1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/fig1_confusion_matrix.png')

## Cell 7 — Figure 2: Feature Importance (Random Forest)

Save to `/content/fig2_feature_importance.png`

In [ ]:
importances = rf.feature_importances_
feat_names  = ['Severity', 'Weather', 'Impact', 'Complaints', 'Accessibility']

# Sort descending
order = np.argsort(importances)[::-1]
sorted_names  = [feat_names[i]  for i in order]
sorted_import = importances[order]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(
    sorted_names[::-1],   # flip so highest is at top
    sorted_import[::-1],
    color=['#1f77b4' if i == 0 else '#aec7e8' for i in range(len(sorted_import)-1, -1, -1)]
)
ax.set_xlabel('Mean Decrease in Gini Impurity', fontsize=11)
ax.set_title('Figure 2: Feature Importance — Random Forest', fontsize=11)
ax.tick_params(axis='both', labelsize=10)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

# Value labels on bars
for bar, val in zip(bars, sorted_import[::-1]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/fig2_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/fig2_feature_importance.png')

## Cell 8 — Table 1: Model Comparison Summary

Prints the table you paste directly into your report.

In [ ]:
summary = pd.DataFrame({
    'Model':           ['Logistic Regression (baseline)', 'Random Forest (primary)'],
    'Train Accuracy':  [f'{acc_lr_train:.3f}', f'{acc_rf_train:.3f}'],
    'Test Accuracy':   [f'{acc_lr_test:.3f}',  f'{acc_rf_test:.3f}'],
    'Train Macro F1':  [f'{f1_lr_train:.3f}',  f'{f1_rf_train:.3f}'],
    'Test Macro F1':   [f'{f1_lr_test:.3f}',   f'{f1_rf_test:.3f}'],
})

print('=== TABLE 1: Model Comparison (copy into report) ===')
print(summary.to_string(index=False))

## Cell 9 — Table 2: Per-Class Report for Random Forest

Formats the sklearn classification_report as a clean DataFrame.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

classes = CLASS_ORDER
prec, rec, f1, sup = precision_recall_fscore_support(
    y_test, y_pred_rf_test, labels=classes
)

table2 = pd.DataFrame({
    'Priority Tier': classes,
    'Precision':     [f'{p:.3f}' for p in prec],
    'Recall':        [f'{r:.3f}' for r in rec],
    'F1-Score':      [f'{f:.3f}' for f in f1],
    'Support':       sup
})

print('=== TABLE 2: Per-Class Results — Random Forest (Test Set) ===')
print(table2.to_string(index=False))
print(f'\nMacro avg F1: {f1_rf_test:.3f}')
print(f'Test Accuracy: {acc_rf_test:.3f}')

## Cell 10 — Fairness: Disparate Impact Ratio (DIR) by Borough

DIR = P(Critical or High | borough) / P(Critical or High | reference borough)
Reference = Manhattan (highest complaint rate, best-connected).
Threshold: DIR ≥ 0.80 passes the four-fifths rule.

In [ ]:
# Attach predictions back to the test set rows
df_test = df.iloc[idx_test].copy()
df_test['predicted_tier'] = y_pred_rf_test

REFERENCE_BOROUGH = 'Manhattan'
FAVORABLE = {'Critical', 'High'}   # favorable = gets repair priority

def favorable_rate(subdf):
    return (subdf['predicted_tier'].isin(FAVORABLE)).mean()

ref_rate = favorable_rate(df_test[df_test['borough'] == REFERENCE_BOROUGH])

dir_rows = []
for borough in ['Brooklyn', 'Queens', 'Manhattan', 'Bronx', 'Staten Island']:
    sub = df_test[df_test['borough'] == borough]
    rate = favorable_rate(sub)
    dire = rate / ref_rate if ref_rate > 0 else float('nan')
    passed = 'PASS ✓' if dire >= 0.80 else 'FAIL ✗'
    dir_rows.append({
        'Borough':           borough,
        'n (test)':          len(sub),
        'P(Critical|High)':  f'{rate:.3f}',
        'DIR':               f'{dire:.3f}',
        'Threshold ≥0.80':   passed
    })

dir_df = pd.DataFrame(dir_rows)
print(f'=== TABLE 3: Disparate Impact Ratio by Borough ===')
print(f'Reference borough: {REFERENCE_BOROUGH} | P(favorable) = {ref_rate:.3f}')
print()
print(dir_df.to_string(index=False))

## Cell 11 — Figure 3: DIR Bar Chart by Borough

In [ ]:
dir_vals    = [float(r['DIR'])     for r in dir_rows]
dir_boroughs = [r['Borough']       for r in dir_rows]
dir_colors  = ['#d62728' if v < 0.80 else '#2ca02c' for v in dir_vals]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(dir_boroughs, dir_vals, color=dir_colors, edgecolor='black', linewidth=0.6)
ax.axhline(0.80, color='black', linestyle='--', linewidth=1.2, label='DIR threshold (0.80)')
ax.axhline(1.00, color='gray',  linestyle=':',  linewidth=0.8, label='Reference (Manhattan)')
ax.set_ylabel('Disparate Impact Ratio', fontsize=11)
ax.set_title('Figure 3: Disparate Impact Ratio by Borough (reference = Manhattan)', fontsize=10)
ax.set_ylim(0, 1.3)
ax.tick_params(axis='both', labelsize=10)
ax.legend(fontsize=9)

# Value labels
for bar, val in zip(bars, dir_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.2f}', ha='center', va='bottom', fontsize=9)

# Color legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ca02c', label='PASS (DIR ≥ 0.80)'),
    Patch(facecolor='#d62728', label='FAIL (DIR < 0.80)'),
]
ax.legend(handles=legend_elements + ax.get_legend_handles_labels()[0][:2], fontsize=9)

plt.tight_layout()
plt.savefig('/content/fig3_dir_by_borough.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: /content/fig3_dir_by_borough.png')

## Cell 12 — Qualitative Results

Pick 3 real incidents from the test set to discuss in the report:
- One where equity adjustment elevated a Bronx/underserved incident (correct)
- One where both models agreed (easy case)
- One failure case to discuss honestly

In [ ]:
df_test['true_tier']      = y_test
df_test['pred_rf']        = y_pred_rf_test
df_test['pred_lr']        = lr.predict(X_test_sc)
df_test['rf_correct']     = df_test['pred_rf'] == df_test['true_tier']

DISPLAY_COLS = ['borough','complaint_type','severity','weather',
                'impact','complaints','accessibility',
                'priority_score','true_tier','pred_rf','pred_lr']

print('=== QUALITATIVE CASE 1: Bronx — equity upweighting in action ===')
bronx_high = df_test[
    (df_test['borough'] == 'Bronx') &
    (df_test['true_tier'].isin(['Critical','High'])) &
    df_test['rf_correct']
].head(1)
print(bronx_high[DISPLAY_COLS].to_string(index=True))

print('\n=== QUALITATIVE CASE 2: Both models agree (easy case) ===')
easy = df_test[
    (df_test['pred_rf'] == df_test['pred_lr']) &
    df_test['rf_correct']
].head(1)
print(easy[DISPLAY_COLS].to_string(index=True))

print('\n=== QUALITATIVE CASE 3: RF failure (honest discussion) ===')
fail = df_test[~df_test['rf_correct']].head(1)
print(fail[DISPLAY_COLS].to_string(index=True))
print('\nNote: severity is simulated (proxy from complaint_type) since Team 1 CV output')
print('is not yet connected. This caps real-world validity of all three cases above.')

## Cell 13 — Overfitting Check

Compare train vs test accuracy for both models.
A large gap means overfit; a small gap is healthy.

In [ ]:
print('=== OVERFITTING CHECK ===')
print(f'Logistic Regression — Train F1: {f1_lr_train:.3f} | Test F1: {f1_lr_test:.3f} | Gap: {f1_lr_train - f1_lr_test:.3f}')
print(f'Random Forest       — Train F1: {f1_rf_train:.3f} | Test F1: {f1_rf_test:.3f} | Gap: {f1_rf_train - f1_rf_test:.3f}')
print()
if (f1_rf_train - f1_rf_test) > 0.10:
    print('WARNING: RF train/test gap > 0.10 — possible overfit.')
    print('Consider: max_depth limit, min_samples_leaf, or more training data.')
else:
    print('Gap is within acceptable range. No significant overfitting detected.')
print()
print('NOTE: Because labels are formula-derived (deterministic function of features),')
print('the RF can learn the formula almost perfectly on training data.')
print('Generalization is still meaningful because test-set scores are held out.')

## Cell 14 — Save All Outputs Summary

Download everything from the Files panel on the left.

In [ ]:
# Save the test-set predictions as CSV for the report appendix
df_test[DISPLAY_COLS].to_csv('/content/citypulse_test_predictions.csv', index=True)

print('All files saved to /content/:')
print('  fig1_confusion_matrix.png    — paste into report')
print('  fig2_feature_importance.png  — paste into report')
print('  fig3_dir_by_borough.png      — paste into report')
print('  citypulse_test_predictions.csv — for qualitative discussion')
print()
print('In Colab: Files panel (left sidebar) → right-click each file → Download')